# ⚠️ Legacy — LDA classifier (superseded)

This notebook trains and applies an older **Linear Discriminant Analysis** classifier. It is kept for reference only. The current, supported classifier is the multi-task CNN in the **`upwins-veg-classifier`** repo, which consumes the reflectance images and training ROIs produced by notebooks 01–03 here.

Paths in this notebook are not wired to `config.yaml`; edit them directly if you want to run it.

In [ ]:
from sklearn import linear_model
import matplotlib.pyplot as plt
from matplotlib import colors
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
import numpy as np
from sklearn import preprocessing
from sklearn.mixture import GaussianMixture
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import numpy as np
#import copy
import spectral
#import time
#import csv
import os
import importlib
import pickle
import utils
from hsiViewer import hsi_viewer_layers as hlv
from hsiViewer import hsi_viewer_ROI as hvr
import matplotlib as mpl
mpl.rcParams['lines.linewidth'] = 0.75

## Step 1. Open the Image

In [ ]:
# data directory
dir = 'data/morven_4000'
fname = 'raw_4000_or_ref.img'
fname_hdr = 'raw_4000_or_ref.hdr'
# Read the image
im = spectral.envi.open(os.path.join(dir,fname_hdr), os.path.join(dir,fname))
im.Arr = im.load().astype(np.float32)
im.mask = im.Arr[:,:,0]!=0
im.ArrList = np.reshape(im.Arr, (im.nrows*im.ncols, im.nbands))
im.wl = np.asarray(im.bands.centers)
wl = im.wl

## Load Spectral Lobrary

In [ ]:
sli_dir = 'D:/SpectralLibrary'
# raw image
fname_sli = os.path.join(sli_dir,'UPWINS_2025-02-12.sli')
fname_sli_hdr = os.path.join(sli_dir,'UPWINS_2025-02-12.hdr')
importlib.reload(utils)
lib = spectral.envi.open(fname_sli_hdr, fname_sli)
names = lib.names
print(np.unique(names))
spectra = lib.spectra
ns, nb_lib = spectra.shape
lib.wl = np.asarray(lib.bands.centers)
wl_lib = lib.wl
print(f'nSpectra, initial nBands = {lib.spectra.shape}')
# resample library to image
resampler = spectral.BandResampler(lib.wl, wl)
spectra = resampler(lib.spectra.T).T
print(f'nSpectra, nBands = {spectra.shape}')

In [ ]:
target_species = ['Panicum_virgatum']

matching_indices_lc = [index for index, item in enumerate(names) if item in target_species]
target_spectra = spectra[matching_indices_lc,:]
target_names = np.array(names)[matching_indices_lc]

print(target_spectra.shape)

In [ ]:
# Read ROI Spectra
with open('Morven_July_Training_ROIs_Img4000.pkl', 'rb') as f:
    training_ROIs = pickle.load(f)
# Get the labels and spectra for the panels
ROI_names = np.asarray(training_ROIs.df.iloc[:,0])
ROI_spectra = np.asarray(training_ROIs.df.iloc[:,4:])
print(f'ROI Names: {ROI_names}')

In [ ]:
print(len(ROI_names))

In [ ]:
roi_classes = np.unique(ROI_names)
print(roi_classes)

In [ ]:
training_ROIs.df

In [ ]:
X = np.vstack((target_spectra, ROI_spectra))
y = np.hstack((target_names, ROI_names))
print(X.shape)
print(y.shape)

# Normalize the data to avoid scaling issues
normalized_X = (X - np.mean(X, axis=1, keepdims=True))/np.std(X, axis=1, keepdims=True)
normalized_imListArr = ((im.ArrList - np.mean(im.ArrList, axis=1, keepdims=True))/(0.00001+np.std(im.ArrList, axis=1, keepdims=True)))
for i in range(normalized_imListArr.shape[1]):
    normalized_imListArr[:,i] = normalized_imListArr[:,i]*np.reshape(im.mask, (im.nrows*im.ncols))

## Train and Apply the Models

In [ ]:
le = preprocessing.LabelEncoder()
le.fit(y)
y_int = le.transform(y)

In [ ]:
# PREDICTING LABELS

clf = LinearDiscriminantAnalysis()
clf.fit(normalized_X, y_int)
LDA_result = clf.predict(normalized_imListArr)
LDA_result_image = np.reshape(LDA_result, (im.nrows, im.ncols))
# Apply the mask
LDA_result_image = (1+LDA_result_image)*im.mask
# Show resulting image
plt.figure(figsize=(12,12))
plt.imshow(LDA_result_image, cmap='jet')

In [ ]:
labels = ['No Data']
print(f'{0}: {['No Data']}')
for i in range(len(np.unique(y_int))):
    print(f'{i+1}: {le.inverse_transform([i])}')
    labels.append(le.inverse_transform([i])[0])


In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(LDA_result_image==7, cmap='jet')

In [ ]:
# PREDICTING PROBABILITIES

clf = LinearDiscriminantAnalysis()
clf.fit(normalized_X, y_int)
LDA_result = clf.predict_proba(normalized_imListArr)
LDA_result_image = np.reshape(LDA_result, (im.nrows, im.ncols, len(np.unique(y))))
print(LDA_result_image.shape)
# Apply the mask
for i in range(len(np.unique(y))):
    LDA_result_image[:,:,i] = LDA_result_image[:,:,i]*im.mask

In [ ]:
# Show resulting image
plt.figure(figsize=(12,12))
plt.imshow(LDA_result_image[:,:,3], cmap='jet')

In [ ]:
hlv.viewer(im, layers={'Morela': LDA_result_image[:,:,6]})

In [ ]:

LDA_result_image = np.reshape(LDA_result_probs, (im.nrows, im.ncols, len(np.unique(y))))